# Tour API 키워드 통합 검색 (`searchKeyword2`) 품질 분석 및 정제 테스트

실시간 통합 검색용 API인 `searchKeyword2`의 원본 데이터 특성을 파악하고, 검색 엔진 및 시뮬레이터 연동에 필요한 정제 규칙을 설계하는 EDA 파일입니다.

### 테스트 항목:
1. **오타 감지 테스트**: 오타가 있는 검색어(예: '해운데') 입력 시 결과 보정 기능 확인.
2. **결측치 분석**: 결측률이 매우 높은 레거시 코드(`cat1/2/3`, `areacode` 등) 식별 및 신분류체계 매핑 적합성 검토.
3. **좌표 보정 및 텍스트 정제**: 위경도가 뒤바뀐 데이터 스왑 보정, 영해 외 이상 좌표 제거, 장소명의 HTML 태그 및 대괄호 설명문 제거 로직을 검증.

In [ ]:
!pip install requests pandas python-dotenv

In [ ]:
import os
import time
import requests
import pandas as pd
from dotenv import load_dotenv

# .env 파일에서 API 키 로드 (상위 폴더 확인)
load_dotenv(dotenv_path="../../.env")
API_KEY = os.getenv("TOUR_API_DECODE_KEY")
BASE_URL = "https://apis.data.go.kr/B551011/KorService2"

print("API 키 로드 상태:", "성공" if API_KEY else "실패 (내부 .env 파일과 변수명을 확인하세요)")

def fetch_api_data(endpoint, base_url=BASE_URL, params=None):
    url = f"{base_url}/{endpoint}"
    default_params = {
        "serviceKey": API_KEY,
        "MobileOS": "ETC",
        "MobileApp": "RouteCheck",
        "_type": "json"
    }
    if params:
        default_params.update(params)
    
    max_retries = 3
    retry_delay = 2  # base delay in seconds
    
    for attempt in range(max_retries + 1):
        try:
            response = requests.get(url, params=default_params, timeout=10)
            
            # [HTTP 429: Too Many Requests 대응]
            if response.status_code == 429:
                if attempt < max_retries:
                    sleep_time = retry_delay * (2 ** attempt)
                    print(f"[WARNING] API 요청 제한(HTTP 429) 발생! {sleep_time}초 대기 후 재시도 ({attempt + 1}/{max_retries})...")
                    time.sleep(sleep_time)
                    continue
                else:
                    print(f"[ERROR] API 요청 제한(HTTP 429) 초과로 요청 최종 실패 ({endpoint})")
                    return pd.DataFrame()
            
            if response.status_code == 200:
                res_json = response.json()
                if 'response' in res_json:
                    header = res_json['response'].get('header', {})
                    result_code = header.get('resultCode')
                    if result_code != '0000':
                        # [공공데이터 포털 서버단 트래픽 제한 에러 대응 (예: '04' 또는 '22')]
                        if result_code in ['04', '22'] and attempt < max_retries:
                            sleep_time = retry_delay * (2 ** attempt)
                            print(f"[WARNING] API 트래픽 초과 에러 [{result_code}] 발생! {sleep_time}초 대기 후 재시도 ({attempt + 1}/{max_retries})...")
                            time.sleep(sleep_time)
                            continue
                        print(f"API 자체 에러 [{result_code}]: {header.get('resultMsg')} ({endpoint})")
                        return pd.DataFrame()
                    
                    body = res_json['response'].get('body', {})
                    items = body.get('items', {})
                    if not items or items == "":
                        return pd.DataFrame()
                    
                    if 'item' in items:
                        item_list = items['item']
                        if isinstance(item_list, dict):
                            item_list = [item_list]
                        return pd.DataFrame(item_list)
            else:
                if attempt < max_retries:
                    print(f"[WARNING] HTTP Error Code: {response.status_code} ({url}). {retry_delay}초 대기 후 재시도...")
                    time.sleep(retry_delay)
                    continue
                else:
                    print(f"HTTP Error Code: {response.status_code} ({url})")
                    return pd.DataFrame()
        except Exception as e:
            if attempt < max_retries:
                print(f"[WARNING] 네트워크 또는 데이터 파싱 실패 ({endpoint}): {e}. {retry_delay}초 대기 후 재시도...")
                time.sleep(retry_delay)
                continue
            else:
                print(f"네트워크 또는 데이터 파싱 최종 실패 ({endpoint}): {e}")
                return pd.DataFrame()
                
    return pd.DataFrame()

## 1. 기본 키워드 검색 조회 기능 검증 (`searchKeyword2`)

In [ ]:
print("\n--- 키워드 검색 조회 (searchKeyword2) ---")

keyword_params = {
    "arrange": "A",
    "keyword": "해운데" # 오타 테스트
}

df_keyword = fetch_api_data(endpoint="searchKeyword2", params=keyword_params)
print("검색 성공 여부 및 검색 크기:", df_keyword.shape)

if not df_keyword.empty:
    print("\n--- 불필요한 레거시 컬럼 제거 ---")
    columns_to_drop = ['areacode', 'sigungucode', 'cat1', 'cat2', 'cat3']
    df_keyword.drop(columns=columns_to_drop, inplace=True, errors='ignore')
    display(df_keyword.head(3))

## 2. [EDA] 데이터 탐색 및 품질 분석 

### 2.1 데이터 기본 정보 및 결측치(Empty String) 분석

In [ ]:
# 대량 데이터 수집 테스트 (키워드: '부산', 500개 행)
eda_params = {
    "arrange": "A",
    "keyword": "부산",
    "numOfRows": 500
}
df_eda = fetch_api_data(endpoint="searchKeyword2", params=eda_params)
print(f"수집된 데이터 크기: {df_eda.shape}")

import numpy as np
# 빈 문자열("") 및 'null' 문자열을 NaN으로 변경하여 결측치 비율 계산
df_eda_nan = df_eda.replace("", np.nan).replace("null", np.nan)

missing_info = pd.DataFrame({
    "결측치 수": df_eda_nan.isnull().sum(),
    "결측 비율 (%)": (df_eda_nan.isnull().sum() / len(df_eda)) * 100
}).sort_values(by="결측 비율 (%)", ascending=False)

display(missing_info.head(10))

### 2.2 범주형 변수 분석 (관광타입 및 신분류체계 비교)

In [ ]:
content_type_map = {
    '12': '관광지',
    '14': '문화시설',
    '15': '축제/공연/행사',
    '25': '여행코스',
    '28': '레포츠',
    '32': '숙박',
    '38': '쇼핑',
    '39': '음식점'
}
df_eda_nan['contenttype_name'] = df_eda_nan['contenttypeid'].map(content_type_map)
print("--- 관광타입별 데이터 분포 ---")
display(df_eda_nan['contenttype_name'].value_counts(dropna=False))

# 신분류체계(lclsSystm3) 한글 매핑 정보 로드
import json
try:
    with open("../data/신분류체계정보_관광타입정보_연계_정의서_exp.json", "r", encoding="utf-8") as f:
        local_cat_map = json.load(f)
    
    # 매핑 성공 개수 확인
    mapped_mask = df_eda_nan['lclsSystm3'].astype(str).str.strip().isin(local_cat_map.keys())
    print(f"\n로컬 신분류 정의서 매핑 성공 개수: {mapped_mask.sum()} / {len(df_eda_nan)} ({mapped_mask.mean() * 100:.2f}%)")
except Exception as e:
    print("로컬 정의서 로드 오류:", e)

### 2.3 위치 좌표(mapx, mapy) 오류 보정 및 영해 외 이상치 정제

In [ ]:
df_eda_nan['mapx_num'] = pd.to_numeric(df_eda_nan['mapx'], errors='coerce')
df_eda_nan['mapy_num'] = pd.to_numeric(df_eda_nan['mapy'], errors='coerce')

print("--- 좌표 통계 정보 ---")
display(df_eda_nan[['mapx_num', 'mapy_num']].describe())

# 1. 위경도 스왑(Swap) 이상치 탐지 및 보정 (Latitude: 33~39, Longitude: 124~132)
swapped_mask = (df_eda_nan['mapx_num'] >= 33.0) & (df_eda_nan['mapx_num'] <= 39.0) & \
               (df_eda_nan['mapy_num'] >= 124.0) & (df_eda_nan['mapy_num'] <= 132.0)

print(f"\n위경도가 거꾸로 뒤바뀐 이상치 감지 개수: {swapped_mask.sum()}개")
if swapped_mask.sum() > 0:
    # 스왑 수행
    temp = df_eda_nan.loc[swapped_mask, 'mapx_num'].copy()
    df_eda_nan.loc[swapped_mask, 'mapx_num'] = df_eda_nan.loc[swapped_mask, 'mapy_num']
    df_eda_nan.loc[swapped_mask, 'mapy_num'] = temp
    print("-> 위경도가 반대로 뒤바뀐 이상치 행들을 성공적으로 스왑하여 보정 완료!")

# 2. 누락 및 대한민국 범위 외부 이상치 필터링
invalid_mask = (
    df_eda_nan['mapx_num'].isna() | df_eda_nan['mapy_num'].isna() |
    (df_eda_nan['mapx_num'] == 0) | (df_eda_nan['mapy_num'] == 0) |
    (df_eda_nan['mapx_num'] < 124.0) | (df_eda_nan['mapx_num'] > 132.0) |
    (df_eda_nan['mapy_num'] < 33.0) | (df_eda_nan['mapy_num'] > 39.0)
)
print(f"좌표 정보 오류(결측, 0, 한국 범위 이탈) 건수: {invalid_mask.sum()}개 (삭제 대상)")

df_cleaned = df_eda_nan[~invalid_mask].copy()
print(f"좌표 정제 후 최종 유효 명소 수: {len(df_cleaned)}")

### 2.4 대표 이미지 및 명소명 텍스트 정제

In [ ]:
# 1. 이미지 정보 분석
image_stats = pd.DataFrame({
    "보유 수": [df_cleaned['firstimage'].notnull().sum(), df_cleaned['firstimage2'].notnull().sum()],
    "보유 비율 (%)": [df_cleaned['firstimage'].notnull().mean() * 100, df_cleaned['firstimage2'].notnull().mean() * 100]
}, index=['대표 이미지 (firstimage)', '썸네일 이미지 (firstimage2)'])
print("--- 이미지 보유 현황 ---")
display(image_stats)

# 2. 장소명(title) HTML 태그 및 특수 기호 정제 테스트
import re
def clean_title(t):
    if not isinstance(t, str):
        return ""
    t = re.sub(r'<[^>]*>', '', t)       # HTML 태그 지우기
    t = re.sub(r'\[[^\]]*\]', '', t)    # 대괄호 명칭 괄호 지우기
    return t.strip()

df_cleaned['title_clean'] = df_cleaned['title'].apply(clean_title)
print("\n--- 장소명 정제 전후 비교 샘플 ---")
display(df_cleaned[['title', 'title_clean']].head(10))

### 2.5 EDA 최종 요약 및 실시간 검색 연동 규칙 정의

1. **결측치 대응**
   - 전화번호(`tel`)의 결측률이 **90% 이상**으로 높으므로, RAG 정보 수립 시 전화번호 기반 조인은 최소화하고 `contentid` 조인을 기본 정책으로 사용합니다.
   - 기존 분류 코드(`cat1`) 대신 결측이 없는 신분류체계 대분류(`lclsSystm1`)와 법정동 지역명(`lDongRegnNm`)을 검색 매핑 메타데이터로 취급합니다.

2. **실시간 검색 이상치 보정**
   - 검색 API `searchKeyword2` 연동 컨트롤러 구현 시, 이번 EDA에서 테스트 완료된 **'위경도 좌표 스왑(Swap) 복구'** 로직과 **'좌표 영해 외 이상치 제거'** 로직을 실시간 데이터 전처리 파이프라인에 탑재하여 경로 연산 안전성을 확보합니다.
   - 명소명에 묻은 HTML 마크업 태그를 청소하기 위해 `clean_title` 가공 정책을 API 리턴 객체에 바인딩하여 프론트엔드로 전달합니다.